# OMDb API demo

Этот ноутбук запрашивает информацию о фильме через OMDb API,
аккуратно обрабатывает ошибки и выводит удобную сводку.

In [1]:
import json
import os

import requests
from dotenv import load_dotenv

In [4]:
title = input("Введите фильм для поиска: ")

try:
    movie_data = fetch_movie_data(title)
    print_movie_summary(movie_data)
except (ValueError, LookupError, RuntimeError) as exc:
    print(f"Ошибка: {exc}")

{
  "Title": "Dune",
  "Year": "1984",
  "Rated": "PG-13",
  "Released": "14 Dec 1984",
  "Runtime": "137 min",
  "Genre": "Action, Adventure, Sci-Fi",
  "Director": "David Lynch",
  "Writer": "Frank Herbert, David Lynch",
  "Actors": "Kyle MacLachlan, Virginia Madsen, Francesca Annis",
  "Plot": "A Duke's son leads desert warriors against the galactic emperor and his father's evil nemesis to free their desert world from the emperor's rule.",
  "Language": "English",
  "Country": "United States, Mexico",
  "Awards": "Nominated for 1 Oscar. 2 wins & 7 nominations total",
  "Poster": "https://m.media-amazon.com/images/M/MV5BMGJlMGM3NDAtOWNhMy00MWExLWI2MzEtMDQ0ZDIzZDY5ZmQ2XkEyXkFqcGc@._V1_SX300.jpg",
  "Ratings": [
    {
      "Source": "Internet Movie Database",
      "Value": "6.3/10"
    },
    {
      "Source": "Rotten Tomatoes",
      "Value": "36%"
    },
    {
      "Source": "Metacritic",
      "Value": "41/100"
    }
  ],
  "Metascore": "41",
  "imdbRating": "6.3",
  "imdbVotes":

In [2]:
load_dotenv()

OMDB_URL = "https://www.omdbapi.com/"
REQUEST_TIMEOUT = 10


def fetch_movie_data(title: str) -> dict:
    api_key = os.getenv("OMDB_API_KEY")
    if not api_key:
        raise ValueError("Ключ OMDB_API_KEY не найден в .env")

    if not title.strip():
        raise ValueError("Название фильма не должно быть пустым")

    try:
        response = requests.get(
            OMDB_URL,
            params={"t": title.strip(), "apikey": api_key},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Ошибка при запросе к OMDb API: {exc}") from exc

    data = response.json()
    if data.get("Response") != "True":
        error_message = data.get("Error", "Неизвестная ошибка OMDb API")
        raise LookupError(f"OMDb API вернул ошибку: {error_message}")

    return data


def print_movie_summary(movie_data: dict) -> None:
    print(json.dumps(movie_data, ensure_ascii=False, indent=2))
    print()
    print(f"Название: {movie_data.get('Title', 'N/A')}")
    print(f"Год: {movie_data.get('Year', 'N/A')}")
    print(f"Жанр: {movie_data.get('Genre', 'N/A')}")
    print(f"Режиссер: {movie_data.get('Director', 'N/A')}")
    print(f"Рейтинг IMDb: {movie_data.get('imdbRating', 'N/A')}")
    print(f"Актеры: {movie_data.get('Actors', 'N/A')}")